In [0]:
# Definir o catálogo e o schema do projeto
catalog = "workspace"
schema = "default"

# Conferir o contexto atual
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

# Listar as tabelas existentes
display(spark.sql("SHOW TABLES IN workspace.default"))

database,tableName,isTemporary
default,olist_customers_dataset,false


In [0]:
from pathlib import Path

volume_path = "/Volumes/workspace/default/ecommerce_raw"

# Listar os arquivos CSV disponíveis no volume
files = [
    f.path
    for f in dbutils.fs.ls(volume_path)
    if f.path.lower().endswith(".csv")
]

if len(files) != 9:
    raise ValueError(
        f"Esperados 9 arquivos CSV, encontrados {len(files)}."
    )

print(f"Arquivos CSV encontrados: {len(files)}")

for file_path in files:
    # Usar o nome do arquivo para gerar o nome da tabela
    table_name = Path(file_path).stem.lower()

    # Ler o CSV preservando os cabeçalhos e inferindo os tipos
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("sep", ",")
        .csv(file_path)
    )

    # Salvar como tabela Delta na camada Bronze
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.default.{table_name}")
    )

    print(f"Tabela criada: workspace.default.{table_name}")

Arquivos CSV encontrados: 9
Tabela criada: workspace.default.olist_customers_dataset
Tabela criada: workspace.default.olist_geolocation_dataset
Tabela criada: workspace.default.olist_order_items_dataset
Tabela criada: workspace.default.olist_order_payments_dataset
Tabela criada: workspace.default.olist_order_reviews_dataset
Tabela criada: workspace.default.olist_orders_dataset
Tabela criada: workspace.default.olist_products_dataset
Tabela criada: workspace.default.olist_sellers_dataset
Tabela criada: workspace.default.product_category_name_translation


In [0]:
%skip
display(spark.sql("SHOW TABLES IN workspace.default"))

database,tableName,isTemporary
default,olist_customers_dataset,false
default,olist_geolocation_dataset,false
default,olist_order_items_dataset,false
default,olist_order_payments_dataset,false
default,olist_order_reviews_dataset,false
default,olist_orders_dataset,false
default,olist_products_dataset,false
default,olist_sellers_dataset,false
default,product_category_name_translation,false


In [0]:
%skip
from pyspark.sql import functions as F

orders = spark.table("workspace.default.olist_orders_dataset")

print(f"Total de pedidos: {orders.count()}")
print(f"Total de colunas: {len(orders.columns)}")

display(orders.limit(5))

Total de pedidos: 99441
Total de colunas: 8


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z
